## Hypothetical Questions

In [1]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime

c:\Users\Mohd Zaid\Desktop\AG05851_3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from groq import Groq

In [4]:
import os
os.environ['API_KEY']=os.getenv('API_KEY')

In [5]:
import os
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

In [6]:
# import os
# os.environ['NIVIDIA_API']=os.getenv('NIVIDIA_API')

In [7]:
from openai import OpenAI

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = os.getenv('GROQ_API_KEY')
)

In [8]:
clients=Groq()

In [9]:
model_name = 'meta/llama-3.1-8b-instruct'

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Mohd Zaid\AppData\Local\Temp\ipykernel_8664\726496232.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [11]:
pdf_folder_location = "tesla-annual-reports"

In [12]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [13]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [14]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [15]:
len(tesla_10k_chunks)

3337

In [16]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [17]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [18]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [19]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [20]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [21]:
i = 0

while i < len(tesla_10k_chunks):
    vectorstore.add_documents( 
        documents=tesla_10k_chunks[i:i+500], 
        ids=["text_" + str(i) for i in range(i, i+500)] 
    )

    i += 500 
    time.sleep(5)

In [22]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [23]:
collection = chromadb_client.get_collection(tesla_10k_collection)

In [24]:
chromadb_client.list_collections()

['hypothetical_questions', 'tesla-10k-2019-to-2023']

In [25]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        'k': 3
    }
)

In [26]:
hypothetical_questions_system_message = """
Generate a list of exactly 3 hypothetical questions that the document presented in the input could be used to answer.
Generate only a list of questions, each question in a new line.
Do not number the questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template = """
<Document>
{document}
</Document>
"""

In [35]:
hypothetical_questions = []

In [36]:
def create_batch_prompt(batch_docs):

    prompt = ""

    for idx, doc in enumerate(batch_docs):

        prompt += f"""
CHUNK {idx+1}

<Document>
{doc.page_content}
</Document>

=========================
"""

    return prompt

In [37]:
batch_system_prompt = """
You will receive multiple document chunks.

For EACH chunk generate exactly 3 hypothetical questions.

Output format:

CHUNK 1
question
question
question

CHUNK 2
question
question
question

Continue for all chunks.

Do not skip any chunk.
Generate exactly 3 questions per chunk.
Do not add explanations.
"""

In [38]:
from langchain_core.documents import Document
import re
import time

In [39]:
BATCH_SIZE = 30

In [40]:
for batch_start in range(0, len(tesla_10k_chunks), BATCH_SIZE):

    batch_docs = tesla_10k_chunks[
        batch_start : batch_start + BATCH_SIZE
    ]

    print(
        f"Processing Chunks "
        f"{batch_start} to "
        f"{batch_start + len(batch_docs) - 1}"
    )

    prompt = create_batch_prompt(batch_docs)

    try:

        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {
                    "role": "system",
                    "content": batch_system_prompt
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        output = response.choices[0].message.content

        # Split by CHUNK headings
        chunk_sections = re.split(
            r"CHUNK\s+\d+",
            output
        )

        chunk_sections = [
            x.strip()
            for x in chunk_sections
            if x.strip()
        ]

        for local_idx, section in enumerate(chunk_sections):

            import re
            questions = re.findall(
            r'[^?]+\?',
            section
            )

            questions = [
            q.strip()
            for q in questions
            ]

            if local_idx >= len(batch_docs):
                continue

            parent_doc = batch_docs[local_idx]

            for question in questions:

                hypothetical_questions.append(
                    Document(
                        page_content=question,
                        metadata={
                            "parent_chunk_id":
                                batch_start + local_idx,
                            "source":
                                parent_doc.metadata.get(
                                    "source"
                                ),
                            "page":
                                parent_doc.metadata.get(
                                    "page"
                                )
                        }
                    )
                )

        print(
            f"Total Questions So Far: "
            f"{len(hypothetical_questions)}"
        )

        time.sleep(2)

    except Exception as e:

        print("ERROR:", e)

        if "429" in str(e):

            print(
                "Rate limit hit. "
                "Sleeping for 60 seconds..."
            )

            time.sleep(60)

        continue

Processing Chunks 0 to 29
Total Questions So Far: 78
Processing Chunks 30 to 59
Total Questions So Far: 168
Processing Chunks 60 to 89
Total Questions So Far: 258
Processing Chunks 90 to 119
Total Questions So Far: 348
Processing Chunks 120 to 149
Total Questions So Far: 438
Processing Chunks 150 to 179
Total Questions So Far: 528
Processing Chunks 180 to 209
Total Questions So Far: 618
Processing Chunks 210 to 239
Total Questions So Far: 708
Processing Chunks 240 to 269
Total Questions So Far: 798
Processing Chunks 270 to 299
Total Questions So Far: 873
Processing Chunks 300 to 329
Total Questions So Far: 963
Processing Chunks 330 to 359
Total Questions So Far: 1053
Processing Chunks 360 to 389
Total Questions So Far: 1143
Processing Chunks 390 to 419
Total Questions So Far: 1233
Processing Chunks 420 to 449
Total Questions So Far: 1323
Processing Chunks 450 to 479
Total Questions So Far: 1413
Processing Chunks 480 to 509
Total Questions So Far: 1503
Processing Chunks 510 to 539
Total

In [48]:
hypothetical_questions_vectorstore = Chroma(
    collection_name="hypothetical_questions_v2",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [49]:
print(hypothetical_questions_vectorstore)

In [50]:
BATCH_SIZE = 5000

for i in range(0, len(hypothetical_questions), BATCH_SIZE):

    batch = hypothetical_questions[i:i+BATCH_SIZE]

    hypothetical_questions_vectorstore.add_documents(
        documents=batch
    )

    print(
        f"Inserted {min(i+BATCH_SIZE, len(hypothetical_questions))}"
        f"/{len(hypothetical_questions)}"
    )

Inserted 5000/9908
Inserted 9908/9908


In [51]:
print(hypothetical_questions_vectorstore._collection.count())

9908


In [52]:
retriever = hypothetical_questions_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

query = "What are Tesla's risk factors?"

hypothetical_questions_retrieved = retriever.invoke(query)

print(len(hypothetical_questions_retrieved))

for doc in hypothetical_questions_retrieved:
    print(doc.page_content)
    print(doc.metadata)
    print("-"*50)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


5
What are the risks related to Tesla's business and industry, and how do they impact the company's financial condition and future results?
{'page': 17, 'parent_chunk_id': 206, 'source': 'tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf'}
--------------------------------------------------
What are the risks related to Tesla's business and industry, and how do they impact the company's financial condition and future results?
{'page': 18, 'parent_chunk_id': 209, 'source': 'tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf'}
--------------------------------------------------
What are the risks related to Tesla's business and industry, and how do they impact the company's financial condition and future results?
{'page': 17, 'parent_chunk_id': 204, 'source': 'tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf'}
--------------------------------------------------
What are the risks related to Tesla's business and industry, and 

In [53]:
chunk_ids = list(
    set(
        [
            f"text_{d.metadata['parent_chunk_id']}"
            for d in hypothetical_questions_retrieved
        ]
    )
)

print(chunk_ids)

['text_206', 'text_204', 'text_209', 'text_205', 'text_207']


In [54]:
retrived_documents = vectorstore.get(
    ids=chunk_ids
)

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


In [55]:
print(len(retrived_documents["documents"]))

5


In [56]:
for i, doc in enumerate(retrived_documents["documents"]):
    print(f"\nCHUNK {i+1}")
    print(doc[:1000])
    print("-" * 100)


CHUNK 1
ITEM	1A.
RISK	FACTORS
You	should	carefully	consider	the	risks	described	below	together	with	the	other	information	set	forth	in	this	report,	which	could	materially
affect	our	business,	financial	condition	and	future	results.	The	risks	described	below	are	not	the	only	risks	facing	our	company.	Risks	and	uncertainties
not	currently	known	to	us	or	that	we	currently	deem	to	be	immaterial	also	may	materially	adversely	affect	our	business,	financial	condition	and
operating	results.
Risks	Related	to	Our	Business	and	Industry
We	have	experienced	in	the	past,	and	may	experience	in	the	future,	delays	or	other	complications	in	the	design,	manufacture,
launch,	and	production	ramp	of	our	vehicles,	energy	products,	and	product	features,	or	may	not	realize	our	manufacturing	cost
targets,	which	could	harm	our	brand,	business,	prospects,	financial	condition	and	operating	results.
We	have	previously	experienced	launch	and	production	ramp	delays	or	other	complications	in	connection	with	new	vehic

In [57]:
context = "\n\n".join(
    retrived_documents["documents"]
)

print(len(context))

8335


In [ ]:
retriever = hypothetical_questions_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [ ]:
# user_query = "What are Tesla's risk factors?"


In [ ]:
# print(len(hypothetical_questions))

In [ ]:
# hypothetical_questions_retrieved = retriever.invoke(user_query)

In [ ]:
# hypothetical_questions_retrieved

In [ ]:
# parent_chunk_id = [d.metadata['parent_chunk_id'] for d in hypothetical_questions_retrieved]
 
# unique_parent_ids = list(dict.fromkeys(parent_chunk_id))
# print(str(unique_parent_ids))

In [ ]:
# retrived_documents = vectorstore.get(
#     ids=list(dict.fromkeys([
#         str(d.metadata['parent_chunk_id'])
#         for d in hypothetical_questions_retrieved
#     ]))
# )

In [ ]:
# print(retrived_documents)

In [ ]:
# context = "\n\n".join(
#     retrived_documents["documents"]
# )

In [ ]:
# print(len(context))

In [ ]:
# final_context_documents=set(context)

In [ ]:
# print(len(final_context_documents))

In [ ]:
# final_context_documents1=set(context)

In [59]:
system_message_hypothetical_question="""
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [63]:
query = "What are Tesla's risk factors?"

user_message = f"""
<Context>
Here are some documents that are relevant to the question mentioned below.

{context}

</Context>

<Question>
{query}
</Question>
"""

In [64]:
print(user_message[:1000])


<Context>
Here are some documents that are relevant to the question mentioned below.

ITEM	1A.
RISK	FACTORS
You	should	carefully	consider	the	risks	described	below	together	with	the	other	information	set	forth	in	this	report,	which	could	materially
affect	our	business,	financial	condition	and	future	results.	The	risks	described	below	are	not	the	only	risks	facing	our	company.	Risks	and	uncertainties
not	currently	known	to	us	or	that	we	currently	deem	to	be	immaterial	also	may	materially	adversely	affect	our	business,	financial	condition	and
operating	results.
Risks	Related	to	Our	Business	and	Industry
We	have	experienced	in	the	past,	and	may	experience	in	the	future,	delays	or	other	complications	in	the	design,	manufacture,
launch,	and	production	ramp	of	our	vehicles,	energy	products,	and	product	features,	or	may	not	realize	our	manufacturing	cost
targets,	which	could	harm	our	brand,	business,	prospects,	financial	condition	and	operating	results.
We	have	previously	experienced	launch	

In [65]:
response = client.chat.completions.create(
    model=model_name,
    messages=[
        {
            "role": "system",
            "content": system_message_hypothetical_question
        },
        {
            "role": "user",
            "content": user_message
        }
    ],
    temperature=0
)

answer = response.choices[0].message.content

print(answer)

Risks Related to Our Business and Industry, Any significant delay or other complication in the production ramp of our current products or the development, manufacture, launch and production ramp of our future products, features and services, including complications associated with expanding our production capacity and supply chain or obtaining or maintaining related regulatory approvals, or inability to manage such ramps cost-effectively, could materially damage our brand, business, prospects, financial condition and operating results.


In [66]:
import json

result = {
    "question_id": "HQ1",
    "user_query": query,
    "retrieved_hypothetical_questions": [
        {
            "hypothetical_question": d.page_content,
            "parent_chunk_id": f"text_{d.metadata['parent_chunk_id']}"
        }
        for d in hypothetical_questions_retrieved
    ],
    "parent_chunks_used": [
        {
            "chunk_id": f"text_{d.metadata['parent_chunk_id']}",
            "source_doc": d.metadata.get("source")
        }
        for d in hypothetical_questions_retrieved
    ],
    "final_answer": answer
}

In [68]:
retrieved_questions = []

for doc in hypothetical_questions_retrieved:

    retrieved_questions.append(
        {
            "hypothetical_question": doc.page_content,
            "parent_chunk_id": f"text_{doc.metadata['parent_chunk_id']}"
        }
    )

print(retrieved_questions[0])

{'hypothetical_question': "What are the risks related to Tesla's business and industry, and how do they impact the company's financial condition and future results?", 'parent_chunk_id': 'text_206'}


In [69]:
parent_chunks = []

seen = set()

for doc in hypothetical_questions_retrieved:

    chunk_id = f"text_{doc.metadata['parent_chunk_id']}"

    if chunk_id not in seen:

        parent_chunks.append(
            {
                "chunk_id": chunk_id,
                "source_doc": doc.metadata["source"]
            }
        )

        seen.add(chunk_id)

print(parent_chunks)

[{'chunk_id': 'text_206', 'source_doc': 'tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf'}, {'chunk_id': 'text_209', 'source_doc': 'tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf'}, {'chunk_id': 'text_204', 'source_doc': 'tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf'}, {'chunk_id': 'text_207', 'source_doc': 'tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf'}, {'chunk_id': 'text_205', 'source_doc': 'tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf'}]


In [70]:
result = {
    "question_id": "HQ1",
    "user_query": query,
    "retrieved_hypothetical_questions": retrieved_questions,
    "parent_chunks_used": parent_chunks,
    "final_answer": answer
}

print(result.keys())

dict_keys(['question_id', 'user_query', 'retrieved_hypothetical_questions', 'parent_chunks_used', 'final_answer'])


In [71]:
import json

def run_hyde_query(query, question_id):

    # Retrieve hypothetical questions
    hypothetical_questions_retrieved = retriever.invoke(query)

    # Parent chunk ids
    chunk_ids = list(
        set(
            [
                f"text_{d.metadata['parent_chunk_id']}"
                for d in hypothetical_questions_retrieved
            ]
        )
    )

    # Retrieve original chunks
    retrived_documents = vectorstore.get(
        ids=chunk_ids
    )

    # Context
    context = "\n\n".join(
        retrived_documents["documents"]
    )

    # Prompt
    user_message = f"""
<Context>
{context}
</Context>

<Question>
{query}
</Question>
"""

    # Final Answer
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": system_message_hypothetical_question
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=0
    )

    answer = response.choices[0].message.content

    # Retrieved Questions
    retrieved_questions = []

    for doc in hypothetical_questions_retrieved:

        retrieved_questions.append(
            {
                "hypothetical_question": doc.page_content,
                "parent_chunk_id": f"text_{doc.metadata['parent_chunk_id']}"
            }
        )

    # Parent Chunks
    parent_chunks = []

    seen = set()

    for doc in hypothetical_questions_retrieved:

        chunk_id = f"text_{doc.metadata['parent_chunk_id']}"

        if chunk_id not in seen:

            parent_chunks.append(
                {
                    "chunk_id": chunk_id,
                    "source_doc": doc.metadata["source"]
                }
            )

            seen.add(chunk_id)

    result = {
        "question_id": question_id,
        "user_query": query,
        "retrieved_hypothetical_questions": retrieved_questions,
        "parent_chunks_used": parent_chunks,
        "final_answer": answer
    }

    return result

In [72]:
queries = [
    "What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?",
    "How should an analyst investigate the relationship between product defects, warranty or service obligations, customer trust, and brand risk?",
    "What evidence helps determine whether future cash flow depends more on capital expenditure discipline, working capital, or operating income?",
    "Which disclosures help evaluate technology, cybersecurity, data, or AI operational risk even if the user does not explicitly say cybersecurity?",
]

In [73]:
all_results = []

for i, query in enumerate(queries):

    print(f"Running HQ{i+1}")

    result = run_hyde_query(
        query=query,
        question_id=f"HQ{i+1}"
    )

    all_results.append(result)

print("Done")

Running HQ1
Running HQ2
Running HQ3
Running HQ4
Done


In [76]:
with open(
    "assignment_output.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_results,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Saved Successfully")

Saved Successfully
